# AgenticML — Experimento 3: California Housing (Regressão)

**Dataset:** California Housing (sklearn)  
**Tipo de problema:** Regressão (valor mediano de imóveis)  
**Objetivo:** Estimar o valor mediano das casas com base nas características demográficas e geográficas do bairro  
**Referência de comparação:** Auto-sklearn (seção 9.1 da spec)  

---
**Relevância:** Valida o caminho de regressão do pipeline, incluindo:
- Agente 5: seleção por R² (critério de seleção), RMSE reportado adicionalmente
- Agente 5: candidatos Linear Regression, Random Forest, XGBoost, SVR
- Agente 3: StandardScaler em features com escalas muito distintas

> ⚠️ **Nota:** O dataset completo tem 20.640 linhas. Para reprodutibilidade mais rápida, 
> foi utilizada uma amostra estratificada de 1.000 linhas (`random_state=42`).

In [2]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing

from core.graph import build_graph
from core.state import initial_state

print('Ambiente configurado.')

Ambiente configurado.


## 1. Dataset — California Housing

| Atributo | Valor |
|----------|-------|
| Instâncias (original) | 20.640 |
| Instâncias (amostra) | 1.000 |
| Features | 8 (MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude) |
| Target | MedHouseVal (valor mediano em $100k) |
| Missing values | Nenhum |
| Fonte | `sklearn.datasets.fetch_california_housing()` |

In [3]:
housing = fetch_california_housing(as_frame=True)
df_full = housing.frame.copy()

# Amostra reprodutível de 1.000 linhas
df = df_full.sample(n=1000, random_state=42).reset_index(drop=True)

print(f'Shape (amostra): {df.shape}')
print(f'Shape (original): {df_full.shape}')
print(f'\nTarget (MedHouseVal) — estatísticas:')
print(df['MedHouseVal'].describe().round(4))
display(df.head())

Shape (amostra): (1000, 9)
Shape (original): (20640, 9)

Target (MedHouseVal) — estatísticas:
count    1000.0000
mean        2.0970
std         1.1643
min         0.3000
25%         1.2552
50%         1.8160
75%         2.7115
max         5.0000
Name: MedHouseVal, dtype: float64


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,1.6812,25.0,4.192201,1.022284,1392.0,3.877437,36.06,-119.01,0.47700
1,2.5313,30.0,5.039384,1.193493,1565.0,2.679795,35.14,-119.46,0.45800
2,3.4801,52.0,3.977155,1.185877,1310.0,1.360332,37.80,-122.44,5.00001
3,5.7376,17.0,6.163636,1.020202,1705.0,3.444444,34.28,-118.72,2.18600
4,3.7250,34.0,5.492991,1.028037,1063.0,2.483645,36.62,-121.93,2.78000


In [4]:
csv_bytes = df.to_csv(index=False).encode('utf-8')
print(f'CSV gerado: {len(csv_bytes):,} bytes')

CSV gerado: 91,700 bytes


## 2. Execução do Pipeline AgenticML

In [5]:
PROMPT = (
    "Estimar o valor mediano das casas (MedHouseVal, em centenas de milhares de dólares) "
    "com base nas características demográficas e geográficas do bloco censitário."
)

state = initial_state(
    dataset_bytes=csv_bytes,
    filename='california_housing.csv',
    prompt=PROMPT,
    problem_type_hint='regression',
)

t0 = time.time()
result = build_graph().invoke(state)
elapsed = time.time() - t0

print(f'\n✓ Pipeline concluído em {elapsed:.2f}s')
print('Status final:', result.get('status'))


✓ Pipeline concluído em 11.79s
Status final: done


## 3. Resultados por Agente

### Agente 1 — Ingestão e Inspeção

In [6]:
inspection = result.get('inspection') or {}
schema = inspection.get('schema', {})

print('Shape detectado:', schema.get('shape'))
print('Encoding:', inspection.get('encoding_detected'))
print('Colunas:', schema.get('columns'))

stats = inspection.get('descriptive_stats', {})
display(pd.DataFrame(stats).T.round(4))

Shape detectado: [1000, 9]
Encoding: utf-8
Colunas: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedHouseVal']


,nulls,mean,std,min,max,median
MedInc,0.0,3.9363,2.0214,0.6831,15.0001,3.5022
HouseAge,0.0,28.9410,12.2895,2.0000,52.0000,30.0000
AveRooms,0.0,5.3906,1.6948,2.0967,36.0755,5.2183
AveBedrms,0.0,1.0755,0.2117,0.5000,5.2736,1.0465
Population,0.0,1414.4890,1033.3068,8.0000,8652.0000,1174.5000
AveOccup,0.0,2.9373,0.7785,1.3130,9.2414,2.8319
Latitude,0.0,35.5259,2.1145,32.5500,41.7900,34.2000
Longitude,0.0,-119.4709,1.9684,-124.1500,-114.5900,-118.4300
MedHouseVal,0.0,2.0970,1.1643,0.3000,5.0000,1.8160


### Agente 2 — Diagnóstico de Qualidade

In [7]:
quality = result.get('quality') or {}
issues = quality.get('issues', [])

print(f"Overall severity: {quality.get('overall_severity')}")
print(f"Issues detectados: {len(issues)}\n")

if issues:
    for i in issues:
        print(f"[{i['severity'].upper()}] {i['type']}")
        print(f"  → {i['alert']}\n")
else:
    print('✓ Nenhum problema de qualidade detectado.')

Overall severity: none
Issues detectados: 0

✓ Nenhum problema de qualidade detectado.


### Agente 3 — Feature Engineering

In [8]:
features = result.get('features') or {}
transforms = features.get('transformations_applied', [])

print(f"Target:   {features.get('target_column')}")
print(f"Features: {features.get('feature_columns')}")
print()
for t in transforms:
    print(f"• {t['type']} em {t['columns']}")
    print(f"  → {t['justification']}")

Target:   MedHouseVal
Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

• standard_scaler em ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
  → As colunas num e9ricas n e3o possuem limites naturais claros e o uso do StandardScaler  e9 apropriado para manter a distribui e7 e3o original dos dados, o que  e9 importante para algoritmos de regress e3o que s e3o sens edveis a outliers.


### Agente 4 — Interpretação de Linguagem Natural

In [9]:
interp = result.get('interpretation') or {}
print(f"Tipo:       {interp.get('problem_type')}")
print(f"Confiança:  {interp.get('confidence')}")
print(f"Divergiu:   {interp.get('diverged_from_user')}")
print(f"\nJustificativa: {interp.get('justification')}")

Tipo:       regression
Confiança:  high
Divergiu:   False

Justificativa: O objetivo do usuário é estimar o valor mediano das casas, que é uma variável numérica contínua representada pela coluna 'MedHouseVal'. As características do dataset incluem várias variáveis numéricas que descrevem aspectos demográficos e geográficos, como 'MedInc', 'HouseAge', 'AveRooms', entre outras. A tarefa de prever um valor numérico contínuo a partir de outras variáveis caracteriza um problema de regressão. Portanto, a sugestão do usuário de que se trata de um problema de regressão está correta.


### Agente 5 — Seleção e Treinamento

Para regressão: critério de seleção = **R²** (melhor que prever a média). RMSE também reportado.

In [10]:
training = result.get('training') or {}

print(f"Modelo selecionado: {training.get('model_selected')}")
print(f"R² (cross-val):     {training.get('metric_value')}")
if 'rmse' in training:
    print(f"RMSE (treino completo): {training.get('rmse')}")
print()

candidates = training.get('candidate_results', [])
df_cands = pd.DataFrame(candidates).sort_values('score', ascending=False)
display(df_cands)

print(f"\nJustificativa: {training.get('justification')}")

Modelo selecionado: SVR
R² (cross-val):     0.715
RMSE (treino completo): 0.5422



,model,score
3,SVR,0.7150
1,Random Forest,0.7059
2,XGBoost,0.6931
0,Linear Regression,0.6515



Justificativa: SVR foi selecionado com r2 de 0.7150 para o problema de regression. Demais candidatos avaliados: Random Forest (0.7059), XGBoost (0.6931), Linear Regression (0.6515).


### Agente 6 — Previsão e Explicação

In [11]:
output = result.get('output') or {}
explanation = output.get('explanation', {})
importances = output.get('feature_importances', [])

print('=== Explicação Técnica ===')
print(explanation.get('technical', ''))

print('\n=== Explicação Acessível ===')
print(explanation.get('accessible', ''))

if importances:
    print('\n=== Importância das Features ===')
    display(pd.DataFrame(importances))

=== Explicação Técnica ===
O pipeline AgenticML foi executado com sucesso, começando com a verificação de qualidade dos dados pelo Agente 2, que não encontrou problemas. O Agente 3 aplicou transformações de normalização usando o `standard_scaler` nas variáveis preditoras, como 'MedInc', 'HouseAge', entre outras, para garantir que todas as features estivessem na mesma escala, o que é crucial para o desempenho de muitos algoritmos de aprendizado de máquina. O Agente 4 confirmou que o problema é de regressão, pois o objetivo é prever o valor mediano das casas ('MedHouseVal'), uma variável numérica contínua.

No estágio de treinamento, o Agente 5 avaliou vários modelos de regressão, incluindo Regressão Linear, Random Forest, XGBoost e SVR. O modelo SVR foi selecionado com base na métrica R², alcançando um valor de 0.715, o que indica que o modelo explica 71.5% da variabilidade dos dados. O RMSE foi de 0.5422, sugerindo um erro médio razoável nas previsões. Embora o SVR não forneça importân

## 4. Resumo

In [ ]:
schema = result.get('inspection') or {}.get('schema', {})
issues = result.get('quality') or {}.get('issues', [])
transforms = result.get('features') or {}.get('transformations_applied', [])

summary = {
    'Experimento':        'California Housing — Regressão',
    'Dataset':            f"{schema.get('shape', [0,0])[0]} linhas × {schema.get('shape', [0,0])[1]} colunas",
    'Tipo de problema':   interp.get('problem_type', 'N/A'),
    'Issues de qualidade': len(issues),
    'Transformações':     len(transforms),
    'Modelo selecionado': training.get('model_selected', 'N/A'),
    'Métrica (seleção)':  'R²',
    'AgenticML R²':       training.get('metric_value', 'N/A'),
    'AgenticML RMSE':     training.get('rmse', 'N/A'),
    'Tempo (s)':          round(elapsed, 2),
}

display(pd.DataFrame([summary]).T.rename(columns={0: 'Valor'}))

,Valor
Experimento,California Housing — Regressão
Dataset,0 linhas × 0 colunas
Tipo de problema,regression
Issues de qualidade,2
Transformações,6
Modelo selecionado,SVR
Métrica (seleção),R²
AgenticML R²,0.715
AgenticML RMSE,0.5422
Auto-sklearn R²,0.705
